In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
print(torch.__version__)

2.14.0+cpu


In [7]:
# Set a "seed" so the random numbers are the same every time you run this.
# Useful for reproducibility — you and I will see the exact same results.
torch.manual_seed(0)

# Create 200 fake data points, each with 4 features (just random numbers for now).
# Think of these 4 numbers as stand-ins for things like packet size, duration, etc.
X = torch.randn(200, 4)

# Create the "correct answers" (labels) for each of those 200 points.
# Our made-up rule: if the 4 features add up to more than 0, label it 1, otherwise 0.
# This gives the model something learnable to figure out.
# .float() converts True/False into 1.0/0.0 (numbers the model can work with).
# .unsqueeze(1) reshapes it from a flat list into a column — PyTorch expects labels
# in this [200, 1] shape to match the model's output shape later.
y = (X.sum(dim=1) > 0).float().unsqueeze(1)

# Print the shapes just to confirm: 200 rows of 4 features, 200 rows of 1 label each.
print(X.shape, y.shape)

torch.Size([200, 4]) torch.Size([200, 1])


In [9]:
class TinyNet(nn.Module):
    # This runs once, when you create the model — it defines the model's building blocks (layers).
    def __init__(self):
        super().__init__()  # required boilerplate — sets up PyTorch's internal machinery

        # First layer: takes in 4 numbers (our features) and outputs 8 numbers.
        # These 8 are the model's own "internal representation" — not meaningful to us directly.
        self.fc1 = nn.Linear(4, 8)

        # Second layer: takes those 8 numbers and boils them down to 1 final number.
        self.fc2 = nn.Linear(8, 1)

        # ReLU is an "activation function" — it adds non-linearity, letting the model
        # learn more than just straight lines/simple math. Common default choice.
        self.relu = nn.ReLU()

        # Sigmoid squashes the final output into a range between 0 and 1 —
        # perfect for "probability of being an attack" style predictions.
        self.sigmoid = nn.Sigmoid()

    # This runs every time you actually pass data through the model (a "forward pass").
    def forward(self, x):
        x = self.relu(self.fc1(x))   # data flows through layer 1, then ReLU
        x = self.sigmoid(self.fc2(x)) # then through layer 2, then Sigmoid
        return x  # final prediction, a number between 0 and 1

# Create an actual instance of the model
model = TinyNet()

# Print it out to see the structure you just defined
print(model)

TinyNet(
  (fc1): Linear(in_features=4, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [11]:
# BCELoss = "Binary Cross-Entropy Loss" — the standard way to measure error
# for yes/no (binary) predictions like ours (attack vs. normal).
# It compares the model's predicted probability (0 to 1) against the true label (0 or 1).
criterion = nn.BCELoss()

# The optimizer is what actually adjusts the model's internal numbers (weights)
# to reduce the loss. Adam is a reliable, commonly-used default choice.
# lr = "learning rate" — how big a step it takes each update. 0.01 is a reasonable default.
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [13]:
# Loop through the whole dataset 50 times (50 "epochs")
for epoch in range(50):

    optimizer.zero_grad()        # Clear out gradients from the previous step —
                                  # PyTorch accumulates them by default, so this resets it.

    outputs = model(X)           # Forward pass: feed all 200 samples through the model,
                                  # get back 200 predictions.

    loss = criterion(outputs, y) # Compare predictions to true labels, get a single
                                  # number representing "how wrong" the model currently is.

    loss.backward()              # Backward pass: PyTorch automatically calculates how much
                                  # each weight in the model contributed to that error.

    optimizer.step()             # Use those calculations to actually nudge the weights
                                  # in the direction that reduces the error.

    # Every 10 epochs, print the current loss so we can watch it improve over time.
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.6545
Epoch 10, Loss: 0.5741
Epoch 20, Loss: 0.4698
Epoch 30, Loss: 0.3572
Epoch 40, Loss: 0.2572


In [15]:
# torch.no_grad() tells PyTorch "don't bother tracking gradients here" —
# we're just checking results now, not training, so this is faster and uses less memory.
with torch.no_grad():

    # Run all 200 samples through the trained model again.
    # If the model's output is > 0.5, we call it a prediction of "1", otherwise "0".
    preds = (model(X) > 0.5).float()

    # Compare predictions to the true labels, and average how often they match —
    # this gives us accuracy as a percentage.
    accuracy = (preds == y).float().mean()

    print(f"Accuracy: {accuracy.item():.2%}")

Accuracy: 96.50%


In [1]:
from datasets import load_dataset

# This downloads the dataset automatically the first time you run it
# (may take a minute or two — it's about 30 MB)
dataset = load_dataset("codymlewis/TON_IoT_network")

# See what splits are available (likely just "train")
print(dataset)

C:\Users\Admin\anaconda3\envs\fl-ids\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 211043/211043 [00:01<00:00, 155987.96 examples/s]


DatasetDict({
    train: Dataset({
        features: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type'],
        num_rows: 211043
    })
    test: Dataset({
        features: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_

In [3]:
import pandas as pd

df = dataset["train"].to_pandas()
df.head()

,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
3,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000113,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
4,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000130,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  object 
 1   src_port                211043 non-null  int64  
 2   dst_ip                  211043 non-null  object 
 3   dst_port                211043 non-null  int64  
 4   proto                   211043 non-null  object 
 5   service                 211043 non-null  object 
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  object 
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_ip_bytes        

In [7]:
df.describe()

,src_port,dst_port,duration,src_bytes,dst_bytes,missed_bytes,src_pkts,src_ip_bytes,dst_pkts,dst_ip_bytes,dns_qclass,dns_qtype,dns_rcode,http_request_body_len,http_response_body_len,http_status_code,label
count,211043.000000,211043.000000,211043.000000,2.110430e+05,2.110430e+05,2.110430e+05,211043.000000,2.110430e+05,211043.000000,2.110430e+05,211043.000000,211043.000000,211043.000000,211043.000000,2.110430e+05,211043.000000,211043.000000
mean,38646.519543,3495.153770,7.700887,2.581136e+05,2.588046e+05,3.443234e+04,9.595220,7.760822e+02,3.846861,1.584687e+03,227.630805,3.610909,0.123989,0.065418,1.449295e+02,0.303905,0.763081
std,19307.271048,10191.624778,564.141946,1.709490e+07,1.802563e+07,5.261621e+06,91.778821,2.229703e+04,330.705796,1.901795e+05,2720.713562,23.797747,0.598804,9.243405,3.047244e+04,8.270377,0.425193
min,1.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000
25%,34608.000000,65.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000,4.800000e+01,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,1.000000
50%,44754.000000,80.000000,0.000170,0.000000e+00,0.000000e+00,0.000000e+00,1.000000,8.200000e+01,1.000000,4.000000e+01,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,1.000000
75%,51133.000000,1253.000000,0.054196,1.300000e+02,8.900000e+01,0.000000e+00,4.000000,4.150000e+02,2.000000,1.340000e+02,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,1.000000
max,65528.000000,65467.000000,93516.929170,3.890855e+09,3.913853e+09,1.854527e+09,24623.000000,6.522626e+06,121942.000000,8.639552e+07,32769.000000,255.000000,5.000000,2338.000000,1.342438e+07,404.000000,1.000000


In [9]:
print(df['label'].value_counts())
print()
print(df['type'].value_counts())

label
1    161043
0     50000
Name: count, dtype: int64

type
normal        50000
backdoor      20000
ddos          20000
dos           20000
injection     20000
password      20000
scanning      20000
ransomware    20000
xss           20000
mitm           1043
Name: count, dtype: int64
